# DimASR: Chain of Thought + Reward-Weighted RL - Laptop Domain

**SemEval-2026 Task 3 - Track A: Subtask 1**

This notebook implements DimASR with:
- **Chain of Thought (CoT)** prompting for interpretable reasoning
- **Reward-Weighted RL** training for improved accuracy

The model learns to generate reasoning before predicting valence-arousal scores.

## Libraries

In [ ]:
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    IN_COLAB = True
except:
    IN_COLAB = False

In [ ]:
if IN_COLAB:
    !pip install transformers datasets evaluate sentencepiece scipy

In [ ]:
import os
import json
import torch
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

if IN_COLAB:
    root_path = 'Enter drive path'
else:
    root_path = '/mnt/f/Jyothi/ITATA'

use_mps = True if torch.backends.mps.is_built() else False
os.chdir(root_path)
print(f"Working directory: {os.getcwd()}")
print(f"MPS available: {use_mps}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
from Imports.data_prep import DatasetLoader
from Imports.utils import T5Generator, RewardWeightedSeq2SeqTrainer
from instructions import InstructionsHandler

## Configuration

In [ ]:
# Task configuration
task_name = 'dimasr_cot_rl'
experiment_name = 'laptop_eng_cot_rl_v1'
model_checkpoint = 'allenai/tk-instruct-base-def-pos'  # or 'google/flan-t5-base'

# Data paths
train_file = './eng_laptop_train_alltasks.jsonl'
dev_file = './eng_laptop_dev_task1.jsonl'

# CoT + RL settings
USE_COT = True           # Enable Chain of Thought prompting
USE_REWARD_WEIGHTED = True  # Enable Reward-Weighted training
RL_WEIGHT = 0.1          # Weight for RL loss component
REWARD_ALPHA = 0.5       # Alpha for reward: exp(-alpha * rmse)

# Model output path
model_out_path = os.path.join('./Models', task_name, f"{model_checkpoint.replace('/', '')}-{experiment_name}")
print('Experiment Name:', experiment_name)
print('Model output path:', model_out_path)
print(f'\nCoT enabled: {USE_COT}')
print(f'Reward-Weighted RL enabled: {USE_REWARD_WEIGHTED}')
print(f'RL weight: {RL_WEIGHT}, Reward alpha: {REWARD_ALPHA}')

## Load Data

In [ ]:
# Load JSONL training data
train_df = DatasetLoader.load_jsonl_data(train_file)
dev_df = DatasetLoader.load_jsonl_data(dev_file)

print(f"Training records: {len(train_df)}")
print(f"Dev records: {len(dev_df)}")
print(f"\nTraining columns: {train_df.columns.tolist()}")
print(f"Dev columns: {dev_df.columns.tolist()}")

In [ ]:
# Display sample data
print("Sample training record:")
print(train_df.iloc[0])

## Setup CoT Instructions

In [ ]:
# Initialize instruction handler
instruct_handler = InstructionsHandler()

# Load CoT instruction set
instruct_handler.load_instruction_set_cot1()

print("DimASR CoT Instructions loaded:")
print(f"- BOS instruction (laptop) length: {len(instruct_handler.dimasr_cot['bos_instruct1'])} chars")
print(f"- Delimiter: '{instruct_handler.dimasr_cot['delim_instruct']}'")
print(f"- EOS: '{instruct_handler.dimasr_cot['eos_instruct']}'")
print("\nNote: EOS ends with 'reasoning:' to prompt model to generate CoT output")

In [ ]:
# Preview instruction template
print("=" * 60)
print("CoT INSTRUCTION TEMPLATE (first 1500 chars):")
print("=" * 60)
print(instruct_handler.dimasr_cot['bos_instruct1'][:1500])
print("...")

## Format Training Data with CoT

In [ ]:
# Create data loader and format training data with CoT
loader = DatasetLoader(train_df_id=None, test_df_id=None)

# Format training data with CoT (generates reasoning templates as labels)
train_formatted = loader.create_data_in_dimasr_cot_format(
    train_df,
    text_col='Text',
    quadruplet_col='Quadruplet',
    bos_instruction=instruct_handler.dimasr_cot['bos_instruct1'],
    delim_instruction=instruct_handler.dimasr_cot['delim_instruct'],
    eos_instruction=instruct_handler.dimasr_cot['eos_instruct'],
    is_train=True
)

print(f"Expanded training samples: {len(train_formatted)}")
print(f"\nColumns: {train_formatted.columns.tolist()}")

In [ ]:
# Display sample formatted CoT data
print("Sample formatted INPUT (with CoT prompt):")
print("-" * 60)
print(train_formatted['text'].iloc[0][-200:])  # Last 200 chars showing the prompt ending
print("\n" + "=" * 60)
print("\nSample LABEL (CoT reasoning + output):")
print("-" * 60)
print(train_formatted['labels'].iloc[0])
print("\n" + "=" * 60)
print(f"\nRaw VA: {train_formatted['va'].iloc[0]}")
print(f"Aspect: {train_formatted['aspect'].iloc[0]}")

## Train/Validation Split

In [ ]:
# Split into training and validation sets
train_split, val_split = train_test_split(train_formatted, test_size=0.1, random_state=42)

# Update loader with split data
loader.train_df_id = train_split.reset_index(drop=True)
loader.val_df_id = val_split.reset_index(drop=True)

print(f"Training samples: {len(train_split)}")
print(f"Validation samples: {len(val_split)}")

## Initialize Model

In [ ]:
# Create T5 Generator
t5_exp = T5Generator(model_checkpoint)
print(f"Model loaded: {model_checkpoint}")
print(f"Device: {t5_exp.device}")

## Tokenize Dataset (with longer output for CoT)

In [ ]:
# Use CoT tokenization with longer max output length (256 vs 64)
id_ds, id_tokenized_ds, ood_ds, ood_tokenized_ds = loader.set_data_for_training_semeval(
    t5_exp.tokenize_function_inputs_cot  # Uses max_length=256 for labels
)

print(f"Tokenized train samples: {len(id_tokenized_ds['train'])}")
print(f"Tokenized validation samples: {len(id_tokenized_ds['validation'])}")

## Training with CoT + Reward-Weighted RL

In [ ]:
# Training arguments - adjusted for CoT + RL
training_args = {
    'output_dir': model_out_path,
    'evaluation_strategy': 'epoch',
    'learning_rate': 3e-5,  # Lower LR for CoT
    'lr_scheduler_type': 'cosine',
    'per_device_train_batch_size': 4,  # Smaller batch for longer sequences
    'per_device_eval_batch_size': 8,
    'num_train_epochs': 8,  # More epochs for CoT
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,
    'push_to_hub': False,
    'eval_accumulation_steps': 1,
    'predict_with_generate': True,
    'use_mps_device': use_mps,
    'gradient_accumulation_steps': 2,  # Effective batch size = 8
}

print("Training configuration (CoT + RL):")
for k, v in training_args.items():
    print(f"  {k}: {v}")

print(f"\nRL Settings:")
print(f"  use_reward_weighted: {USE_REWARD_WEIGHTED}")
print(f"  rl_weight: {RL_WEIGHT}")
print(f"  reward_alpha: {REWARD_ALPHA}")

In [ ]:
# Train the model with Reward-Weighted RL
model_trainer = t5_exp.train(
    id_tokenized_ds,
    use_reward_weighted=USE_REWARD_WEIGHTED,
    rl_weight=RL_WEIGHT,
    reward_alpha=REWARD_ALPHA,
    **training_args
)

## Inference & Evaluation

In [ ]:
# Load trained model for inference
t5_exp = T5Generator(model_out_path)
print(f"Loaded trained model from: {model_out_path}")

In [ ]:
# Re-tokenize for inference
id_ds, id_tokenized_ds, _, _ = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs_cot)

# Get predictions on validation set (with longer max_length for CoT)
val_pred = t5_exp.get_labels(id_tokenized_ds, sample_set='validation', batch_size=8, max_length=256)
val_true = [label.strip() for label in id_ds['validation']['labels']]

print(f"Generated {len(val_pred)} predictions")

In [ ]:
# Calculate metrics using CoT-aware evaluation
metrics = t5_exp.get_metrics_regression_cot(val_true, val_pred)

print("\n" + "="*50)
print("VALIDATION METRICS (CoT + RL)")
print("="*50)
print(f"RMSE_VA (Official): {metrics['RMSE_VA']:.4f}")
print(f"PCC Valence:        {metrics['PCC_V']:.4f}")
print(f"PCC Arousal:        {metrics['PCC_A']:.4f}")
print(f"PCC Average:        {metrics['PCC_avg']:.4f}")
print(f"RMSE Valence:       {metrics['RMSE_V']:.4f}")
print(f"RMSE Arousal:       {metrics['RMSE_A']:.4f}")
print("="*50)

In [ ]:
# Display sample CoT predictions with reasoning
print("\nSample CoT Predictions (with reasoning):")
print("=" * 70)
for i in range(min(5, len(val_pred))):
    va_pred, reasoning = t5_exp.parse_cot_va_prediction(val_pred[i], return_reasoning=True)
    true_va = t5_exp.parse_cot_va_prediction(val_true[i])
    
    print(f"\n--- Sample {i+1} ---")
    print(f"True VA: {true_va}")
    print(f"Pred VA: {va_pred}")
    print(f"Reasoning: {reasoning[:150]}..." if len(reasoning) > 150 else f"Reasoning: {reasoning}")

## Generate Predictions for Dev Set (Submission)

In [ ]:
# Format dev data for inference (no labels)
dev_formatted = loader.create_data_in_dimasr_cot_format(
    dev_df,
    text_col='Text',
    aspect_col='Aspect',
    bos_instruction=instruct_handler.dimasr_cot['bos_instruct1'],
    delim_instruction=instruct_handler.dimasr_cot['delim_instruct'],
    eos_instruction=instruct_handler.dimasr_cot['eos_instruct'],
    is_train=False
)

print(f"Dev samples to predict: {len(dev_formatted)}")

In [ ]:
# Create loader for dev predictions
loader_dev = DatasetLoader(train_df_id=dev_formatted, test_df_id=None)
dev_ds, dev_tokenized_ds, _, _ = loader_dev.set_data_for_training_semeval(t5_exp.tokenize_function_inputs_cot)

# Get predictions (longer max_length for CoT output)
dev_predictions = t5_exp.get_labels(dev_tokenized_ds, sample_set='train', batch_size=8, max_length=256)
print(f"Generated {len(dev_predictions)} predictions for dev set")

In [ ]:
# Format output as required JSONL (extract VA from CoT output)
def format_output_jsonl_cot(dev_formatted_df, predictions, output_path, t5_model):
    """
    Format CoT predictions into required output JSONL format.
    Extracts VA from CoT reasoning output.
    """
    results = {}
    
    for idx, row in dev_formatted_df.iterrows():
        record_id = row['ID']
        aspect = row['aspect']
        # Parse VA from CoT prediction
        va_pred = t5_model.parse_cot_va_prediction(predictions[idx])
        
        if record_id not in results:
            results[record_id] = []
        
        results[record_id].append({
            'Aspect': aspect,
            'VA': va_pred
        })
    
    # Write output JSONL
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for record_id, aspect_vas in results.items():
            output_record = {
                'ID': record_id,
                'Aspect_VA': aspect_vas
            }
            f.write(json.dumps(output_record, ensure_ascii=False) + '\n')
    
    print(f"Output saved to: {output_path}")
    return results

# Generate output file
output_path = './predictions/pred_eng_laptop_cot_rl.jsonl'
results = format_output_jsonl_cot(dev_formatted, dev_predictions, output_path, t5_exp)

In [ ]:
# Display sample predictions with CoT reasoning
print("\nSample predictions for submission (with reasoning):")
print("=" * 70)
for i, (record_id, aspect_vas) in enumerate(list(results.items())[:3]):
    print(f"\nID: {record_id}")
    for j, av in enumerate(aspect_vas[:2]):  # Show first 2 aspects
        print(f"  Aspect: {av['Aspect']:20} VA: {av['VA']}")
        # Show reasoning for first prediction
        if j == 0:
            idx = i * len(aspect_vas) + j
            if idx < len(dev_predictions):
                _, reasoning = t5_exp.parse_cot_va_prediction(dev_predictions[idx], return_reasoning=True)
                print(f"    Reasoning: {reasoning[:100]}..." if len(reasoning) > 100 else f"    Reasoning: {reasoning}")

In [ ]:
# Verify output file format
print("\nFirst 3 lines of output file:")
print("-" * 60)
with open(output_path, 'r') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line.strip())

## Summary

This notebook has:
1. Loaded the DimASR training data with **Chain of Thought** instruction templates
2. Generated synthetic reasoning labels for training
3. Trained the model with **Reward-Weighted RL** to focus on harder samples
4. Evaluated using CoT-aware metrics extraction
5. Generated predictions with interpretable reasoning

The output file `predictions/pred_eng_laptop_cot_rl.jsonl` is ready for submission.

### CoT + RL Benefits:
- **Interpretability**: Model generates reasoning explaining its VA prediction
- **Improved Accuracy**: RL weighting focuses learning on harder samples
- **Error Analysis**: Reasoning output helps debug prediction errors